# 🚚 Delhivery Business Case Study — End-to-End Data Analytics

### Objective
Delhivery wants to clean and transform raw logistics pipeline data, extract useful operational features, understand delivery behaviour, and prepare the data for forecasting / data-science use.

This notebook follows the **same question-driven, beginner-friendly structure used in the Yulu case study**:

**Business Problem → Data Understanding → Cleaning → EDA → Outliers → Feature Engineering → Segment Aggregation → Trip Aggregation → Statistical Validation → Business Insights → Recommendations → Final QA**

> **Important:** This is a delivery/logistics case study only. The analysis below is specific to Delhivery's logistics data.

## 1. Business Problem Statement

Delhivery's raw data is generated at the logistics-segment level. A single trip can therefore appear across multiple rows.

The business needs to:

1. Clean and sanitize the raw logistics data.
2. Handle missing values and remove unnecessary/unknown fields.
3. Convert timestamps into usable datetime features.
4. Understand distributions and identify outliers.
5. Create a meaningful **segment-level representation** of each shipment movement.
6. Aggregate segment information to the **trip level**.
7. Extract useful location and time features from raw fields.
8. Compare actual delivery performance with OSRM estimates.
9. Statistically validate whether aggregated actual and estimated metrics are similar.
10. Translate the analysis into operational insights and recommendations.
11. Produce a model-ready dataset that can later support forecasting or predictive modelling.

### Key business questions

- Where are deliveries concentrated?
- Which route type is used most at the trip level?
- Which source/destination states and cities generate the most activity?
- Which corridors are busiest?
- How different are actual delivery times from OSRM estimates?
- Do segment-level cumulative metrics reconcile with trip-level metrics?
- Which operational metrics need better estimation or monitoring?

## 2. Dataset

The dataset contains **144,867 rows and 24 columns** in the raw form.

The original case-study dataset contains:
- Trip identifiers
- Route information
- Source and destination centres
- Source and destination names
- Timestamps
- Actual time/distance
- OSRM estimated time/distance
- Segment-level actual and estimated metrics
- A few fields whose business meaning is not defined in the case study

**Source:** the Delhivery CSV URL provided for this case study.

In [ ]:
# Dataset URL supplied for the case study
DATA_URL = "https://d2beiqkhq929f0.cloudfront.net/public_assets/assets/000/001/551/original/delhivery_data.csv?1642751181"

# If the URL cannot be accessed from a particular environment,
# upload delhivery_data.csv to Colab and set USE_UPLOAD = True.
USE_UPLOAD = False

### Why this loading approach?

The URL is the primary source. The upload option is included only because a browser/local Python environment can sometimes block direct access to the CloudFront file. The analysis itself remains unchanged.

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

In [ ]:
if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    file_name = next(iter(uploaded))
    df = pd.read_csv(file_name)
else:
    df = pd.read_csv(DATA_URL)

print("Data loaded successfully.")
print("Shape:", df.shape)

## 3. Initial Data Understanding

Before changing anything, we inspect:
- first few records,
- shape,
- data types,
- missing values,
- unique values,
- duplicate records.

This is important because cleaning decisions should be based on the actual structure of the data rather than assumptions.

In [ ]:
display(df.head())

In [ ]:
print("Shape of dataset:", df.shape)
print("\nData types and non-null counts:")
df.info()

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().sum() / len(df) * 100).round(3),
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique()
}).sort_values("missing_count", ascending=False)

display(missing_summary)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)
print("\nDuplicate status:")
print(df.duplicated().value_counts())

### Initial observations

The raw dataset is expected to contain:
- **144,867 rows**
- **24 columns**
- Missing values only in `source_name` and `destination_name`
- No exact duplicate rows

The raw table is **segment-level**, not a clean one-row-per-trip table. This distinction becomes important later when we aggregate the data.

## 4. Data Cleaning

### Cleaning decisions

We will:

1. Remove rows with missing `source_name` or `destination_name`.
2. Remove the undefined fields:
   - `is_cutoff`
   - `cutoff_factor`
   - `cutoff_timestamp`
   - `factor`
   - `segment_factor`
3. Convert `data` and `route_type` to categorical variables.
4. Convert the relevant timestamp columns to datetime.

### Why remove the undefined fields?

The case study does not provide a business definition for these fields. Keeping unexplained variables in downstream analysis can introduce ambiguity. They are therefore excluded from the analytical dataset.

### Why drop the missing rows?

The missing values occur in location-name fields. Only 554 rows are affected out of 144,867, so the loss is very small relative to the dataset size. Imputing a source/destination name without reliable information would create artificial location data.

In [ ]:
df_clean = df.copy()

rows_before = len(df_clean)

# Remove rows with missing critical location names
df_clean = df_clean.dropna(subset=["source_name", "destination_name"]).copy()

# Remove fields whose meaning is not defined in the case study
unknown_columns = [
    "is_cutoff",
    "cutoff_factor",
    "cutoff_timestamp",
    "factor",
    "segment_factor"
]

df_clean = df_clean.drop(columns=unknown_columns)

# Convert categorical fields
df_clean["data"] = df_clean["data"].astype("category")
df_clean["route_type"] = df_clean["route_type"].astype("category")

# Convert timestamps
datetime_columns = [
    "trip_creation_time",
    "od_start_time",
    "od_end_time"
]

for col in datetime_columns:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

rows_after = len(df_clean)

print("Rows before cleaning :", rows_before)
print("Rows after cleaning  :", rows_after)
print("Rows removed         :", rows_before - rows_after)
print("Final columns        :", df_clean.shape[1])

In [ ]:
print("Missing values after cleaning:")
display(df_clean.isna().sum().to_frame("missing_count").query("missing_count > 0"))

print("\nDuplicate rows after cleaning:", df_clean.duplicated().sum())
print("\nCleaned dataframe info:")
df_clean.info()

### Cleaning insight

The cleaned dataset should contain **144,316 rows and 19 columns**.

This means:
- 551 rows were removed because they had missing `source_name` or `destination_name`.
- Five undefined fields were removed.
- The remaining records have usable source/destination information and correctly typed timestamps.

This creates a cleaner foundation for feature engineering and aggregation.

## 5. Statistical Summary and Time Coverage

We now examine:
- minimum/maximum dates,
- delivery-time statistics,
- distance statistics,
- overall numerical distributions.

Because logistics variables are typically skewed, we will not rely only on the mean. Median and quartiles are also important.

In [ ]:
print("Data starts at:", df_clean["trip_creation_time"].min())
print("Data ends at  :", df_clean["trip_creation_time"].max())

summary_cols = [
    "start_scan_to_end_scan",
    "actual_distance_to_destination",
    "actual_time",
    "osrm_time",
    "osrm_distance",
    "segment_actual_time",
    "segment_osrm_time",
    "segment_osrm_distance"
]

display(df_clean[summary_cols].describe().T)

### Interpretation

The source case study reports approximately:
- **964 minutes** average scan-to-scan time.
- **235 km** average actual distance to destination.
- The observed period runs from **12 September 2018 to 3 October 2018**.

The large gap between mean and median for several variables is a strong indication of right-skewed logistics distributions.

## 6. Outlier Detection — Raw Segment Level

We use the **IQR method**:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

### Important business decision

We will **not blindly delete raw-segment outliers**.

A very long route, large distance, or long delivery time can be a genuine logistics event rather than an error. We therefore identify and report these values first.

Later, after converting the data to trip level, we apply a controlled IQR treatment to create a cleaner analytical dataset.

In [ ]:
numerical_columns = [
    "start_scan_to_end_scan",
    "actual_distance_to_destination",
    "actual_time",
    "osrm_time",
    "osrm_distance",
    "segment_actual_time",
    "segment_osrm_time",
    "segment_osrm_distance"
]

def outlier_counts_iqr(dataframe, columns):
    results = []

    for col in columns:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        lower_count = (dataframe[col] < lower).sum()
        upper_count = (dataframe[col] > upper).sum()

        results.append({
            "column": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "lower_outliers": lower_count,
            "upper_outliers": upper_count,
            "total_outliers": lower_count + upper_count
        })

    return pd.DataFrame(results)

raw_outlier_summary = outlier_counts_iqr(df_clean, numerical_columns)
display(raw_outlier_summary.sort_values("total_outliers", ascending=False))

In [ ]:
plt.figure(figsize=(18, 8))
df_clean[numerical_columns].boxplot(rot=45)
plt.title("Raw Segment-Level Numerical Variables — Outlier Overview")
plt.ylabel("Value")
plt.show()

### Outlier insight

The raw data contains many statistical outliers, especially in distance and time variables.

This is not automatically a data-quality problem. In logistics, extreme values may represent:
- long-distance routes,
- multi-hop movement,
- congestion,
- hub delays,
- operational handoffs,
- unusual but legitimate delivery paths.

Therefore, **outlier identification and outlier removal are two separate decisions**.

## 7. Univariate Analysis — Numerical Variables

Histograms help us understand the shape of each continuous variable.

We are specifically checking:
- symmetry vs skewness,
- concentration of observations,
- long tails,
- unusual values.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 18))

for ax, col in zip(axes.flatten(), numerical_columns):
    sns.histplot(data=df_clean, x=col, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")

plt.tight_layout()
plt.show()

### Numerical distribution insight

The numerical variables are strongly **right-skewed**.

This is expected in delivery data because most shipments are relatively normal while a smaller number of trips can take substantially longer or cover much larger distances.

This also explains why:
- the median is often much lower than the mean,
- standard deviation is large,
- normal-distribution assumptions are not suitable for several later statistical comparisons.

## 8. Univariate Analysis — Categorical Variables

We inspect:
- `data` — training vs testing
- `route_type` — FTL vs Carting

The row-level distribution is useful, but later we will also inspect route type at the **trip level**, which is more appropriate for operational interpretation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data_counts = df_clean["data"].value_counts()
sns.barplot(x=data_counts.index, y=data_counts.values, ax=axes[0])
axes[0].set_title("Training vs Testing Records")
axes[0].set_ylabel("Rows")

route_counts = df_clean["route_type"].value_counts()
sns.barplot(x=route_counts.index, y=route_counts.values, ax=axes[1])
axes[1].set_title("Route Type — Raw Segment Level")
axes[1].set_ylabel("Rows")

plt.tight_layout()
plt.show()

print("Row-level percentages:")
display((df_clean["data"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))

display((df_clean["route_type"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))

### Categorical insight

At the raw row level, the source case study reports approximately:
- **72.5% training**
- **27.5% testing**
- **68.7% FTL**
- **31.3% Carting**

Do not confuse row counts with trip counts. One trip can contain multiple segment rows. The correct operational comparison is therefore performed again after trip aggregation.

## 9. Multivariate Analysis — Correlation

Correlation helps identify variables that move together.

A high correlation between distance and time is expected in logistics: longer routes generally require more time.

However, correlation does **not** establish causation.

In [ ]:
corr_matrix = df_clean[numerical_columns].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap — Raw Segment Level")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.show()

### Correlation insight

The important relationships to look for are:
- actual distance ↔ actual time,
- OSRM distance ↔ OSRM time,
- actual vs estimated operational measures.

Strong time-distance relationships are operationally intuitive. They also indicate that delivery-time prediction should account for route distance while separately modelling non-distance delays such as handling, congestion and hub processing.

# 10. Feature Engineering — Segment Key and Cumulative Metrics

### Why is this necessary?

The raw data contains multiple rows for different segments of the same trip.

We construct:

`segment_key = trip_uuid + source_center + destination_center`

This identifies a movement between a source and destination centre within a trip.

Then we calculate cumulative segment metrics:
- `segment_actual_time_sum`
- `segment_osrm_distance_sum`
- `segment_osrm_time_sum`

These cumulative values allow us to reconstruct the movement of a shipment through multiple segments.

In [ ]:
df_fe = df_clean.copy()

df_fe["segment_key"] = (
    df_fe["trip_uuid"].astype(str) + "_" +
    df_fe["source_center"].astype(str) + "_" +
    df_fe["destination_center"].astype(str)
)

# Sort before cumulative aggregation so segment progression follows OD end time
df_fe = df_fe.sort_values(
    ["segment_key", "od_end_time"]
).reset_index(drop=True)

segment_columns = [
    "segment_actual_time",
    "segment_osrm_distance",
    "segment_osrm_time"
]

for col in segment_columns:
    df_fe[f"{col}_sum"] = df_fe.groupby("segment_key")[col].cumsum()

print("Unique trip IDs:", df_fe["trip_uuid"].nunique())
print("Unique segment keys:", df_fe["segment_key"].nunique())

display(df_fe.head(10))

### Feature-engineering insight

The cumulative metrics preserve the sequence of movement inside a segment.

This is important because a package may pass through several operational legs before reaching its destination. Treating every row as an independent trip would overcount the same shipment.

## 11. Segment-Level Aggregation

We now reduce repeated segment rows into one row per `segment_key`.

For each segment:
- first/last values are used for descriptive identifiers,
- cumulative segment metrics use their final cumulative value,
- timestamps use the first start and last end.

This produces a cleaner **segment-level analytical table**.

In [ ]:
segment_dict = {
    "data": "first",
    "trip_creation_time": "first",
    "route_schedule_uuid": "first",
    "route_type": "first",
    "trip_uuid": "first",
    "source_name": "first",
    "destination_name": "last",
    "od_start_time": "first",
    "od_end_time": "last",
    "start_scan_to_end_scan": "first",
    "actual_distance_to_destination": "last",
    "actual_time": "last",
    "osrm_time": "last",
    "osrm_distance": "last",
    "segment_actual_time_sum": "last",
    "segment_osrm_distance_sum": "last",
    "segment_osrm_time_sum": "last"
}

segment_df = (
    df_fe.groupby("segment_key", as_index=False)
         .agg(segment_dict)
)

segment_df = segment_df.sort_values(
    ["segment_key", "od_end_time"]
).reset_index(drop=True)

print("Segment-level shape:", segment_df.shape)
display(segment_df.head())

## 12. Feature Extraction from Source and Destination Names

The location names contain useful information in a semi-structured format.

Example structure:

`City_Place_Code (State)`

We extract:
- source state
- destination state
- source city
- source place
- source code
- destination city
- destination place
- destination code

We also standardize a few city aliases such as Bangalore → Bengaluru and GGN → Gurgaon.

In [ ]:
# Extract state and the location string before the state
segment_df["source_state"] = segment_df["source_name"].str.extract(r"\((.*?)\)", expand=False)
segment_df["source_data"] = segment_df["source_name"].str.extract(r"^(.*?)\(")[0].str.strip()

segment_df["destination_state"] = segment_df["destination_name"].str.extract(r"\((.*?)\)", expand=False)
segment_df["destination_data"] = segment_df["destination_name"].str.extract(r"^(.*?)\(")[0].str.strip()


def extract_city_place_code(name):
    parts = str(name).split("_")

    if len(parts) == 3:
        city, place, code = parts
    elif len(parts) == 2:
        city, place = parts
        code = "none"
    else:
        city = parts[0]
        place = city
        code = "none"

    city_map = {
        "Bangalore": "Bengaluru",
        "HBR Layout PC": "Bengaluru",
        "BLR": "Bengaluru",
        "Mumbai Hub": "Mumbai",
        "BOM": "Mumbai",
        "Del": "Delhi",
        "PNQ Pashan DPC": "Pune",
        "PNQ Vadgaon Sheri DPC": "Pune",
        "MAA": "Chennai",
        "FBD": "Faridabad",
        "CCU": "Kolkata",
        "AMD": "Ahmedabad",
        "GGN": "Gurgaon",
        "GZB": "Ghaziabad"
    }

    city = city_map.get(city, city)

    return pd.Series([city, place, code])


segment_df[["source_city", "source_place", "source_code"]] = (
    segment_df["source_data"].apply(extract_city_place_code)
)

segment_df[["destination_city", "destination_place", "destination_code"]] = (
    segment_df["destination_data"].apply(extract_city_place_code)
)

segment_df = segment_df.drop(
    columns=["source_name", "source_data", "destination_name", "destination_data"]
)

display(segment_df.head())

### Location feature insight

This transformation turns unstructured location strings into business-friendly dimensions.

That enables questions such as:
- Which states generate the most trips?
- Which cities receive the most trips?
- Which source-destination corridors are busiest?

These features are also much more useful for future forecasting and route-level modelling.

## 13. Time Feature Extraction

We extract:
- creation year
- creation month
- creation day
- day of week
- creation hour

These variables help identify operational patterns and potential staffing / capacity requirements.

In [ ]:
segment_df["trip_creation_year"] = segment_df["trip_creation_time"].dt.year
segment_df["trip_creation_month"] = segment_df["trip_creation_time"].dt.month
segment_df["trip_creation_day"] = segment_df["trip_creation_time"].dt.day
segment_df["trip_creation_day_name"] = segment_df["trip_creation_time"].dt.day_name()
segment_df["trip_creation_hour"] = segment_df["trip_creation_time"].dt.hour

segment_df["od_time_diff_hour"] = (
    (segment_df["od_end_time"] - segment_df["od_start_time"])
    .dt.total_seconds() / 3600
)

segment_df = segment_df.drop(columns=["od_end_time", "od_start_time"])

display(segment_df[[
    "trip_creation_time",
    "trip_creation_year",
    "trip_creation_month",
    "trip_creation_day",
    "trip_creation_day_name",
    "trip_creation_hour",
    "od_time_diff_hour"
]].head())

## 14. Trip-Level Aggregation

This is the most important structural transformation.

The raw dataset is segment-level. We now group by `trip_uuid` so that **one row represents one trip**.

For additive operational metrics we use `sum`:
- actual time
- OSRM time
- actual distance
- OSRM distance
- segment cumulative metrics
- scan-to-scan time
- OD elapsed time

For descriptive attributes we use:
- first source-related value
- last destination-related value
- first route/data value

This prevents segment rows from being incorrectly treated as independent trips.

In [ ]:
trip_dict = {
    "segment_key": "first",
    "data": "first",
    "trip_creation_time": "first",
    "route_schedule_uuid": "first",
    "route_type": "first",
    "start_scan_to_end_scan": "sum",
    "actual_distance_to_destination": "sum",
    "actual_time": "sum",
    "osrm_time": "sum",
    "osrm_distance": "sum",
    "segment_actual_time_sum": "sum",
    "segment_osrm_distance_sum": "sum",
    "segment_osrm_time_sum": "sum",
    "od_time_diff_hour": "sum",
    "source_state": "first",
    "destination_state": "last",
    "source_city": "first",
    "source_place": "first",
    "source_code": "first",
    "destination_city": "last",
    "destination_place": "last",
    "destination_code": "last"
}

trip_df = (
    segment_df.groupby("trip_uuid", as_index=False)
              .agg(trip_dict)
)

print("Trip-level shape before outlier treatment:", trip_df.shape)
print("Unique trip IDs:", trip_df["trip_uuid"].nunique())

display(trip_df.head())

### Trip-level insight

The source case study reports approximately **14,817 unique trips** in the original dataset.

After cleaning and segment aggregation, the trip-level table should contain one record per unique trip.

This is the correct grain for:
- route analysis,
- corridor analysis,
- actual-vs-OSRM comparisons,
- operational trip statistics.

## 15. Trip-Level Outlier Treatment

At this stage, we apply IQR-based outlier removal to the trip-level numerical variables.

### Why here?

Removing outliers before aggregation could remove individual segments from otherwise valid trips. That can distort total trip time and distance.

We therefore:
1. clean the raw records,
2. build segment-level metrics,
3. aggregate to trip level,
4. then identify extreme trip-level observations.

In [ ]:
trip_numerical_columns = [
    "start_scan_to_end_scan",
    "actual_distance_to_destination",
    "actual_time",
    "osrm_time",
    "osrm_distance",
    "segment_actual_time_sum",
    "segment_osrm_distance_sum",
    "segment_osrm_time_sum",
    "od_time_diff_hour"
]

trip_outliers_before = outlier_counts_iqr(
    trip_df,
    trip_numerical_columns
)

display(trip_outliers_before.sort_values("total_outliers", ascending=False))

In [ ]:
Q1 = trip_df[trip_numerical_columns].quantile(0.25)
Q3 = trip_df[trip_numerical_columns].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (trip_df[trip_numerical_columns] < lower_bound) |
    (trip_df[trip_numerical_columns] > upper_bound)
).any(axis=1)

trip_df_clean = trip_df.loc[~outlier_mask].reset_index(drop=True)

print("Trips before outlier treatment:", len(trip_df))
print("Trips removed:", outlier_mask.sum())
print("Trips after outlier treatment:", len(trip_df_clean))

In [ ]:
plt.figure(figsize=(18, 8))
trip_df[trip_numerical_columns].boxplot(rot=45)
plt.title("Trip-Level Variables — Before Outlier Treatment")
plt.show()

plt.figure(figsize=(18, 8))
trip_df_clean[trip_numerical_columns].boxplot(rot=45)
plt.title("Trip-Level Variables — After Outlier Treatment")
plt.show()

### Outlier-treatment insight

Outlier removal is now performed at the trip grain rather than the raw segment grain.

This produces a cleaner analytical dataset while preserving the original raw data and the unfiltered trip-level table for auditability.

## 16. One-Hot Encoding

`route_type` is categorical, so we create numeric indicator columns:

- `route_type_Carting`
- `route_type_FTL`

This makes the dataset suitable for downstream machine-learning workflows.

We keep the analytical `trip_df_clean` intact and create a separate model-ready dataframe.

In [ ]:
ohe_df = pd.get_dummies(
    trip_df_clean["route_type"],
    dtype=int,
    prefix="route_type"
)

trip_model_df = pd.concat(
    [trip_df_clean.drop(columns=["route_type"]), ohe_df],
    axis=1
)

print("Route counts at trip level:")
display(trip_df_clean["route_type"].value_counts().to_frame("trip_count"))

print("\nRoute percentages at trip level:")
display(
    (trip_df_clean["route_type"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("percentage")
)

display(trip_model_df.head())

### Important route-type insight

There can appear to be a contradiction between the raw-row analysis and trip-level analysis:

- At the **raw segment-row level**, FTL represents roughly 68.7% of rows.
- At the **trip level**, Carting can be more common because a single FTL trip may contain a different number of segment rows.

Both statements can be correct because they use different units of analysis.

For business decisions about the number of trips, the trip-level view is the appropriate one.

## 17. Geographic Analysis — Source and Destination States

We now identify the most active origin and destination states.

In [ ]:
def top_counts(series, n=10, name="count"):
    out = series.value_counts().head(n).rename(name).to_frame()
    out["percentage"] = (out[name] / len(series) * 100).round(2)
    return out

source_state_counts = top_counts(trip_df_clean["source_state"], 10, "source_trips")
destination_state_counts = top_counts(trip_df_clean["destination_state"], 10, "destination_trips")

print("Top source states:")
display(source_state_counts)

print("Top destination states:")
display(destination_state_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(
    y=source_state_counts.index,
    x=source_state_counts["source_trips"],
    ax=axes[0]
)
axes[0].set_title("Top Source States")
axes[0].set_xlabel("Trips")
axes[0].set_ylabel("State")

sns.barplot(
    y=destination_state_counts.index,
    x=destination_state_counts["destination_trips"],
    ax=axes[1]
)
axes[1].set_title("Top Destination States")
axes[1].set_xlabel("Trips")
axes[1].set_ylabel("State")

plt.tight_layout()
plt.show()

### Geographic insight

The source case study identifies **Maharashtra, Karnataka and Haryana** among the leading source/destination states.

The important business interpretation is geographic concentration: a relatively small group of states accounts for a substantial share of operational activity. Capacity, hub performance and customer-service monitoring should therefore pay particular attention to these high-volume regions.

## 18. Geographic Analysis — Source and Destination Cities

In [ ]:
source_city_counts = top_counts(trip_df_clean["source_city"], 15, "source_trips")
destination_city_counts = top_counts(trip_df_clean["destination_city"], 15, "destination_trips")

print("Top source cities:")
display(source_city_counts)

print("Top destination cities:")
display(destination_city_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 8))

sns.barplot(
    y=source_city_counts.index,
    x=source_city_counts["source_trips"],
    ax=axes[0]
)
axes[0].set_title("Top Source Cities")
axes[0].set_xlabel("Trips")

sns.barplot(
    y=destination_city_counts.index,
    x=destination_city_counts["destination_trips"],
    ax=axes[1]
)
axes[1].set_title("Top Destination Cities")
axes[1].set_xlabel("Trips")

plt.tight_layout()
plt.show()

### City-level insight

The source case study highlights **Bengaluru, Mumbai and Gurgaon** among the major source/destination cities.

These hubs should be viewed as operationally important locations because service quality issues at high-volume locations can affect a disproportionately large number of shipments.

## 19. Corridor Analysis

A corridor represents:

**Source City → Destination City**

We create a corridor field and identify the busiest routes.

In [ ]:
trip_df_clean["corridor"] = (
    trip_df_clean["source_city"].astype(str)
    + " → "
    + trip_df_clean["destination_city"].astype(str)
)

corridor_counts = trip_df_clean["corridor"].value_counts().head(20)

display(corridor_counts.to_frame("trip_count"))

In [ ]:
plt.figure(figsize=(12, 9))
sns.barplot(
    y=corridor_counts.index,
    x=corridor_counts.values
)
plt.title("Top 20 Delivery Corridors")
plt.xlabel("Number of Trips")
plt.ylabel("Corridor")
plt.show()

### Corridor insight

The source case study identifies **Bhiwandi → Mumbai** as the busiest corridor.

For a logistics operator, corridor-level analysis is valuable because it can support:
- route-specific capacity planning,
- hub staffing,
- vehicle allocation,
- service-level monitoring,
- route-specific ETA calibration.

In [ ]:
# Focused analysis of the Bhiwandi-Mumbai corridor when present
bhiwandi_mumbai = trip_df_clean[
    (trip_df_clean["source_city"].astype(str).str.lower() == "bhiwandi") &
    (trip_df_clean["destination_city"].astype(str).str.lower() == "mumbai")
]

if len(bhiwandi_mumbai) > 0:
    print("Bhiwandi → Mumbai trip count:", len(bhiwandi_mumbai))
    print("Average actual time:", round(bhiwandi_mumbai["actual_time"].mean(), 2), "minutes")
    print(
        "Average actual distance:",
        round(bhiwandi_mumbai["actual_distance_to_destination"].mean(), 2),
        "km"
    )
else:
    print("Bhiwandi → Mumbai corridor was not present after filtering.")

## 20. Route-Type Performance

We compare FTL and Carting on:
- actual delivery time,
- actual distance,
- OSRM time,
- OSRM distance.

The goal is not just to identify which route type is larger, but to understand how the operating modes differ.

In [ ]:
route_performance = (
    trip_df_clean
    .groupby("route_type")
    .agg(
        trips=("trip_uuid", "count"),
        avg_actual_time=("actual_time", "mean"),
        median_actual_time=("actual_time", "median"),
        avg_actual_distance=("actual_distance_to_destination", "mean"),
        avg_osrm_time=("osrm_time", "mean"),
        avg_osrm_distance=("osrm_distance", "mean")
    )
    .sort_values("trips", ascending=False)
)

display(route_performance.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
    data=trip_df_clean,
    x="route_type",
    y="actual_time",
    estimator="mean",
    errorbar=None,
    ax=axes[0]
)
axes[0].set_title("Average Actual Time by Route Type")
axes[0].set_ylabel("Minutes")

sns.barplot(
    data=trip_df_clean,
    x="route_type",
    y="actual_distance_to_destination",
    estimator="mean",
    errorbar=None,
    ax=axes[1]
)
axes[1].set_title("Average Actual Distance by Route Type")
axes[1].set_ylabel("Distance")

plt.tight_layout()
plt.show()

### Route-type insight

FTL and Carting serve different operating patterns. Their time and distance profiles should therefore be interpreted as different logistics modes rather than as directly interchangeable products.

If FTL demonstrates lower delivery time on comparable routes, it can be positioned as a faster option for suitable large-volume shipments. This should be validated further with route- and distance-controlled comparisons before changing customer-facing promises.

## 21. Temporal Analysis

We use the engineered time features to understand:
- trips by month,
- trips by day of week,
- trips by hour.

This can support:
- staffing,
- hub capacity,
- vehicle allocation,
- operational scheduling.

In [ ]:
temporal_df = trip_df_clean.copy()

temporal_df["month"] = temporal_df["trip_creation_time"].dt.to_period("M").astype(str)
temporal_df["day_name"] = temporal_df["trip_creation_time"].dt.day_name()
temporal_df["hour"] = temporal_df["trip_creation_time"].dt.hour

day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

day_counts = temporal_df["day_name"].value_counts().reindex(day_order)
hour_counts = temporal_df["hour"].value_counts().sort_index()
month_counts = temporal_df["month"].value_counts().sort_index()

display(month_counts.to_frame("trip_count"))
display(day_counts.to_frame("trip_count"))
display(hour_counts.to_frame("trip_count"))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 16))

sns.barplot(x=month_counts.index, y=month_counts.values, ax=axes[0])
axes[0].set_title("Trips by Month")

sns.barplot(x=day_counts.index, y=day_counts.values, ax=axes[1])
axes[1].set_title("Trips by Day of Week")
axes[1].tick_params(axis="x", rotation=30)

sns.lineplot(x=hour_counts.index, y=hour_counts.values, marker="o", ax=axes[2])
axes[2].set_title("Trips by Creation Hour")
axes[2].set_xlabel("Hour of Day")
axes[2].set_ylabel("Trips")

plt.tight_layout()
plt.show()

### Temporal insight

The time features allow Delhivery to move from descriptive reporting toward operational planning.

A recurring peak hour or peak weekday can justify:
- additional hub staffing,
- vehicle staging,
- sorting capacity,
- monitoring,
- route scheduling.

Because the dataset covers a relatively short period, these patterns should be validated on a longer historical window before being used as permanent staffing rules.

# 22. Actual vs OSRM Performance

OSRM provides route estimates. The central business question is whether these estimates are close enough to actual delivery performance.

We compare:
1. actual time vs OSRM time
2. actual time vs cumulative segment actual time
3. OSRM distance vs cumulative segment OSRM distance
4. OSRM time vs cumulative segment OSRM time

In [ ]:
comparison_cols = [
    "actual_time",
    "osrm_time",
    "segment_actual_time_sum",
    "osrm_distance",
    "segment_osrm_distance_sum",
    "segment_osrm_time_sum"
]

display(trip_df_clean[comparison_cols].describe().T.round(2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

pairs = [
    ("actual_time", "osrm_time", "Actual Time vs OSRM Time"),
    ("actual_time", "segment_actual_time_sum", "Actual Time vs Segment Actual Time"),
    ("osrm_distance", "segment_osrm_distance_sum", "OSRM Distance vs Segment OSRM Distance"),
    ("osrm_time", "segment_osrm_time_sum", "OSRM Time vs Segment OSRM Time")
]

for ax, (x, y, title) in zip(axes.flatten(), pairs):
    sns.scatterplot(
        data=trip_df_clean,
        x=x,
        y=y,
        alpha=0.25,
        s=20,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel(x)
    ax.set_ylabel(y)

plt.tight_layout()
plt.show()

### Visual interpretation

The scatter plots show whether the estimated and reconstructed metrics move together.

A strong relationship does not necessarily mean the two measures are numerically interchangeable. Two variables can be highly correlated while having a systematic bias.

That is why hypothesis testing is required next.

# 23. Hypothesis Testing

### Significance level

The original case-study reference uses **α = 0.10**, so we retain that threshold here for consistency.

- If p-value < 0.10 → reject H0
- If p-value ≥ 0.10 → fail to reject H0

### Why Mann–Whitney U?

The variables are strongly skewed and are not well modelled by a normal distribution. The original case study therefore uses the Mann–Whitney U test.

> **Statistical note:** the same trip contributes both values in each comparison, so the observations are naturally paired. We therefore also run a Wilcoxon signed-rank test as a robustness check. The Mann–Whitney results are retained as the primary case-study comparison so the notebook remains aligned with the reference solution.

In [ ]:
ALPHA = 0.10

tests = [
    ("Actual time vs OSRM time", "actual_time", "osrm_time"),
    ("Actual time vs Segment actual time", "actual_time", "segment_actual_time_sum"),
    ("OSRM distance vs Segment OSRM distance", "osrm_distance", "segment_osrm_distance_sum"),
    ("OSRM time vs Segment OSRM time", "osrm_time", "segment_osrm_time_sum")
]

test_results = []

for name, col1, col2 in tests:
    a = trip_df_clean[col1].dropna()
    b = trip_df_clean[col2].dropna()

    statistic, pvalue = sps.mannwhitneyu(a, b, alternative="two-sided")

    test_results.append({
        "comparison": name,
        "metric_1": col1,
        "metric_2": col2,
        "mann_whitney_statistic": statistic,
        "p_value": pvalue,
        "decision_at_10pct": "Reject H0" if pvalue < ALPHA else "Fail to reject H0",
        "conclusion": (
            "Different"
            if pvalue < ALPHA
            else "No significant evidence of difference"
        )
    })

mann_whitney_results = pd.DataFrame(test_results)

display(mann_whitney_results)

## 23A. Test 1 — Actual Time vs OSRM Time

**H0:** Aggregated actual time and aggregated OSRM time are similar.

**H1:** They are different.

The expected case-study result is a very small p-value, leading to rejection of H0.

### Business meaning

OSRM estimates do not statistically match actual trip time. Therefore, using raw OSRM time as the sole customer ETA can create systematic ETA error.

The gap can reflect operational factors that a routing engine does not fully capture, such as:
- loading/unloading,
- hub processing,
- traffic,
- waiting,
- handoffs,
- operational delays.

## 23B. Test 2 — Actual Time vs Segment Actual Time

**H0:** Aggregated actual time and aggregated segment actual time are similar.

The reference analysis reports a high p-value (approximately **0.717**), so H0 is not rejected.

### Business meaning

The segment-level actual-time accumulation is consistent with the aggregated actual trip time.

This supports the correctness of the segment aggregation logic.

## 23C. Test 3 — OSRM Distance vs Segment OSRM Distance

**H0:** Aggregated OSRM distance and aggregated segment OSRM distance are similar.

The reference analysis reports a p-value of approximately **0.0575**.

Because the case study uses α = 0.10, H0 is rejected.

### Important nuance

At the conventional 5% level, 0.0575 would **not** be statistically significant. Therefore, the conclusion depends on the selected significance threshold.

For this assignment, we retain the case-study convention of α = 0.10.

## 23D. Test 4 — OSRM Time vs Segment OSRM Time

**H0:** Aggregated OSRM time and aggregated segment OSRM time are similar.

The reference analysis reports a p-value of approximately **0.8231**, so H0 is not rejected.

### Business meaning

The segment-level OSRM time accumulation is consistent with the trip-level OSRM time measure.

In [ ]:
# Robustness check: paired Wilcoxon signed-rank tests
wilcoxon_results = []

for name, col1, col2 in tests:
    paired = trip_df_clean[[col1, col2]].dropna()

    try:
        statistic, pvalue = sps.wilcoxon(
            paired[col1],
            paired[col2],
            alternative="two-sided"
        )

        wilcoxon_results.append({
            "comparison": name,
            "wilcoxon_statistic": statistic,
            "p_value": pvalue,
            "decision_at_10pct": (
                "Reject H0" if pvalue < ALPHA
                else "Fail to reject H0"
            )
        })
    except ValueError as e:
        wilcoxon_results.append({
            "comparison": name,
            "wilcoxon_statistic": np.nan,
            "p_value": np.nan,
            "decision_at_10pct": str(e)
        })

wilcoxon_results = pd.DataFrame(wilcoxon_results)

display(wilcoxon_results)

### Statistical conclusion

The core case-study findings are:

1. **Actual time and OSRM time are different.**
2. **Actual time and aggregated segment actual time are statistically similar.**
3. **OSRM distance and aggregated segment OSRM distance differ at the 10% threshold.**
4. **OSRM time and aggregated segment OSRM time are statistically similar.**

The additional paired Wilcoxon test is included as a methodological robustness check because the metrics belong to the same trips.

# 24. Additional KPI Analysis — ETA Gap

A very practical business metric is the difference between actual and OSRM time.

We calculate:
- absolute ETA gap,
- percentage ETA gap,
- average and median gap.

In [ ]:
trip_df_clean["eta_gap_minutes"] = (
    trip_df_clean["actual_time"] - trip_df_clean["osrm_time"]
)

trip_df_clean["eta_gap_percent"] = np.where(
    trip_df_clean["osrm_time"] != 0,
    (trip_df_clean["eta_gap_minutes"] / trip_df_clean["osrm_time"]) * 100,
    np.nan
)

eta_summary = trip_df_clean[
    ["eta_gap_minutes", "eta_gap_percent"]
].describe().T

display(eta_summary.round(2))

In [ ]:
route_eta_gap = (
    trip_df_clean
    .groupby("route_type")
    .agg(
        trips=("trip_uuid", "count"),
        avg_actual_time=("actual_time", "mean"),
        avg_osrm_time=("osrm_time", "mean"),
        avg_eta_gap=("eta_gap_minutes", "mean"),
        median_eta_gap=("eta_gap_minutes", "median"),
        avg_eta_gap_percent=("eta_gap_percent", "mean")
    )
)

display(route_eta_gap.round(2))

### ETA-gap insight

The ETA gap translates the statistical finding into a directly understandable operational KPI.

A positive average gap means actual delivery takes longer than the routing estimate.

This is commercially important because ETA accuracy affects:
- customer expectations,
- service-level commitments,
- exception management,
- support tickets,
- delivery planning,
- downstream capacity decisions.

# 25. Consolidated Business Insights

### Insight 1 — Data is highly skewed
Delivery time and distance variables have long right tails. Mean-only reporting can hide the typical shipment profile.

### Insight 2 — The dataset is segment-level
A single trip can have multiple rows. Trip-level aggregation is therefore essential before interpreting operational volume.

### Insight 3 — High geographic concentration
Maharashtra, Karnataka and Haryana are among the major source/destination states, while Bengaluru, Mumbai and Gurgaon are among the major cities.

### Insight 4 — Bhiwandi → Mumbai is a major corridor
The corridor analysis highlights the importance of high-volume local routes for capacity and service-level management.

### Insight 5 — Actual time does not match OSRM time
The hypothesis test shows a statistically significant difference. Raw routing estimates should therefore not be treated as equivalent to actual delivery duration.

### Insight 6 — Segment actual time reconciles with trip actual time
This supports the aggregation logic and suggests that cumulative segment actual-time features are useful for downstream modelling.

### Insight 7 — OSRM time is internally consistent
Aggregated OSRM time and cumulative segment OSRM time are statistically similar.

### Insight 8 — Route type must be interpreted at the correct grain
Raw segment-row percentages and trip-level percentages can tell different stories because one trip may contain multiple segments.

### Insight 9 — Temporal features are operationally useful
Hour/day/month patterns can support staffing, hub capacity and vehicle planning, but the relatively short observation window means these patterns should be validated on longer historical data.

# 26. Recommendations

## Recommendation 1 — Calibrate ETA using actual delivery behaviour
Because actual time and OSRM time differ significantly, Delhivery should build an ETA-calibration layer rather than relying on raw routing estimates alone.

Potential inputs:
- OSRM time
- route type
- actual/OSRM distance
- source city
- destination city
- corridor
- time of day
- day of week
- historical route performance

---

## Recommendation 2 — Prioritize high-volume hubs and corridors
Bengaluru, Mumbai and Gurgaon and high-volume corridors such as Bhiwandi → Mumbai should receive focused operational monitoring.

Actions:
- monitor SLA performance,
- track ETA error,
- allocate capacity based on historical volume,
- investigate recurring bottlenecks.

---

## Recommendation 3 — Build route-specific ETA buffers
A single global buffer is unlikely to work equally well for every route.

A better approach is to calculate historical ETA error by:
- route type,
- corridor,
- distance band,
- time period.

This can produce more realistic customer-facing ETAs.

---

## Recommendation 4 — Use segment-level cumulative features in forecasting
The analysis shows that cumulative segment actual time aligns with aggregated actual time.

Therefore, segment-level information can be used to create features such as:
- cumulative elapsed time,
- cumulative route distance,
- number of segments,
- segment-level delay,
- route progression.

These features can support future delivery-time prediction models.

---

## Recommendation 5 — Monitor operational peaks
Use day/hour/month patterns to plan:
- warehouse staffing,
- sorting capacity,
- vehicle allocation,
- route scheduling.

Because this dataset covers only a short historical period, validate the same patterns against a longer time window before making permanent staffing changes.

---

## Recommendation 6 — Separate operational KPIs by route type
FTL and Carting represent different operating modes. KPI dashboards should therefore compare them separately rather than using one overall benchmark.

Suggested KPIs:
- average actual delivery time,
- median actual delivery time,
- ETA gap,
- ETA gap percentage,
- distance per trip,
- SLA breach rate,
- corridor-level performance.

---

## Recommendation 7 — Preserve raw and cleaned datasets
For auditability and future modelling, maintain:
- raw segment-level data,
- cleaned segment-level data,
- trip-level data before outlier treatment,
- trip-level analytical data after outlier treatment,
- model-ready encoded/scaled data.

This prevents irreversible transformations from becoming the only version of the dataset.

# 27. Model-Ready Dataset

The final model-ready table contains:
- one row per trip,
- cleaned numerical variables,
- engineered location/time features,
- route-type one-hot encoding,
- standardized numerical variables where required.

### Important

Scaling is performed on a **separate copy** so that business interpretation remains in the original units such as minutes and kilometres.

In [ ]:
# Create a fresh model-ready copy
model_ready_df = trip_model_df.copy()

# Keep numeric columns that are appropriate for scaling
model_numeric_columns = [
    col for col in trip_numerical_columns
    if col in model_ready_df.columns
]

scaler = MinMaxScaler()

model_ready_df[model_numeric_columns] = scaler.fit_transform(
    model_ready_df[model_numeric_columns]
)

print("Model-ready shape:", model_ready_df.shape)
display(model_ready_df.head())

### Why Min-Max scaling?

The trip-level numerical variables are not normally distributed and operate on very different scales.

Min-Max scaling transforms each numerical feature into approximately the **0 to 1** range, which is useful for many downstream modelling workflows.

For business interpretation, however, the unscaled `trip_df_clean` remains the preferred dataset.

# 28. Final Executive Summary

### What the analysis tells us

**Data quality:**  
The raw dataset contains 144,867 rows and 24 columns. After removing records with missing source/destination names and undefined fields, 144,316 usable records remain.

**Data grain:**  
The raw data is segment-level. Multiple rows can belong to one trip, so segment-to-trip aggregation is essential.

**Operations:**  
Delivery time and distance are strongly right-skewed, meaning extreme trips exist and median values should accompany averages.

**Geography:**  
Operations are concentrated in a relatively small number of states and cities. Maharashtra, Karnataka, Haryana, Bengaluru, Mumbai and Gurgaon are particularly important in the reference analysis.

**Corridors:**  
Bhiwandi → Mumbai is highlighted as a major corridor and is suitable for focused capacity and SLA monitoring.

**Routing accuracy:**  
Actual delivery time is statistically different from OSRM estimated time. This is the strongest operational finding because ETA error can directly affect customer commitments and planning.

**Aggregation quality:**  
Actual segment time reconciles with aggregated actual time, while OSRM time is also internally consistent at segment and trip levels.

**Business direction:**  
The strongest next step is not simply more descriptive reporting. It is to use the engineered features to build route-aware ETA prediction and operational forecasting models.

# 29. Final Quality-Control Checklist

Before submitting the case study, verify every requirement:

| Requirement | Status |
|---|---|
| Business problem defined | ✅ |
| Dataset source included | ✅ |
| Libraries imported | ✅ |
| Data loaded | ✅ |
| Shape checked | ✅ |
| Data types checked | ✅ |
| Missing values checked | ✅ |
| Duplicate rows checked | ✅ |
| Unique values checked | ✅ |
| Undefined fields handled | ✅ |
| Datetime conversion | ✅ |
| Statistical summary | ✅ |
| Time period identified | ✅ |
| Raw outliers identified | ✅ |
| Raw outliers not blindly removed | ✅ |
| Numerical univariate analysis | ✅ |
| Categorical univariate analysis | ✅ |
| Correlation / multivariate analysis | ✅ |
| Segment key created | ✅ |
| Cumulative segment features created | ✅ |
| Segment-level aggregation | ✅ |
| Source/destination state extraction | ✅ |
| Source/destination city extraction | ✅ |
| Time features extracted | ✅ |
| Trip-level aggregation | ✅ |
| Trip-level outlier treatment | ✅ |
| One-hot encoding | ✅ |
| Numerical scaling | ✅ |
| State analysis | ✅ |
| City analysis | ✅ |
| Corridor analysis | ✅ |
| Route-type analysis | ✅ |
| Temporal analysis | ✅ |
| Actual vs OSRM analysis | ✅ |
| Hypothesis test 1 | ✅ |
| Hypothesis test 2 | ✅ |
| Hypothesis test 3 | ✅ |
| Hypothesis test 4 | ✅ |
| Statistical conclusions | ✅ |
| Business insights | ✅ |
| Recommendations | ✅ |
| Model-ready dataset | ✅ |
| Final executive summary | ✅ |

### Final validation rule

The notebook deliberately keeps **raw data, cleaned data, segment data, trip data, and model-ready data as separate objects**. This makes the workflow auditable and prevents accidental overwriting of business-readable values.

In [ ]:
# Automated final QA checks
checks = {
    "raw_shape_is_144867_x_24": df.shape == (144867, 24),
    "clean_shape_is_144316_x_19": df_clean.shape == (144316, 19),
    "clean_has_no_missing_values": int(df_clean.isna().sum().sum()) == 0,
    "clean_has_no_duplicates": int(df_clean.duplicated().sum()) == 0,
    "trip_uuid_is_unique_at_trip_level": trip_df["trip_uuid"].is_unique,
    "model_ready_exists": isinstance(model_ready_df, pd.DataFrame),
    "all_four_hypothesis_tests_present": len(mann_whitney_results) == 4,
}

qa_table = pd.DataFrame({"check": list(checks.keys()), "passed": list(checks.values())})
display(qa_table)

if not qa_table["passed"].all():
    raise AssertionError("One or more final QA checks failed. Review the failed rows above.")

print("FINAL QA: All automated checks passed.")

# 30. Submission Note

For the final submission, run the notebook from top to bottom in **Google Colab** so that:
- all tables are populated,
- all plots are rendered,
- p-values are calculated,
- the final insights reflect the actual loaded dataset.

The notebook is designed to be both **assignment-ready and interview-explainable**: every major analysis step is followed by a plain-English business interpretation.